## STAGE 1 - IMPORTS AND CONFIGURATION

In [1]:
# !pip install instructor
# !pip install ragas
# !pip uninstall langchain-ollama -y
# !pip install langchain-ollama==0.1.3
# %pip install --upgrade langchain-ollama
# %pip install langchain-community==0.4.1
# !pip install -U ragas

In [2]:
# %pip freeze > requirements.txt

In [3]:
import os
import time
import logging
import numpy as np
import pandas as pd
import nltk

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate

from bert_score import BERTScorer
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score

from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.metrics import (
    GEval,
    AnswerRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    FaithfulnessMetric,
)
from deepeval.models import OllamaModel
from tqdm.auto import tqdm

# Model configuration
MODEL = "llama3.2"
EVALUATION_MODEL_NAME = "llama3.2"

# Ollama runs locally; these are estimated API costs, not electricity/hardware costs.
INPUT_PRICE = 0.0
OUTPUT_PRICE = 0.0

# File paths
INPUT_CSV = "ai_teacher_20_questions.csv"
PDF_FILE = "HopeAI_Teaching.pdf"
OUTPUT_CSV = "RAG_LLM_FINAL_EVALUATION.csv"

# Validate required input files before loading
for path in (INPUT_CSV, PDF_FILE):
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"Required file not found: {path}. Place it in the notebook's working directory."
        )

# Load and validate input data
df = pd.read_csv(INPUT_CSV)
required_columns = ["User Query", "Teacher Response"]
missing = set(required_columns) - set(df.columns)
if missing:
    raise ValueError(f"Missing CSV columns: {sorted(missing)}")

df = df.dropna(subset=required_columns).copy()
for col in required_columns:
    df[col] = df[col].astype(str).str.strip()
df = df[(df["User Query"] != "") & (df["Teacher Response"] != "")].reset_index(drop=True)
if df.empty:
    raise ValueError("The input CSV has no usable rows after removing empty questions/answers.")

# Initialize generation and evaluation models
llama_model = ChatOllama(model=MODEL, temperature=0, timeout=120)
evaluation_model = OllamaModel(
    model=EVALUATION_MODEL_NAME,
    base_url="http://localhost:11434",
    temperature=0,
    generation_kwargs={
        "num_predict": 512
    }
)

logging.getLogger("deepeval").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

print(f"Questions loaded: {len(df)}")
print(f"Generation model: {MODEL}")
print(f"Evaluation model: {EVALUATION_MODEL_NAME}")


Questions loaded: 20
Generation model: llama3.2
Evaluation model: llama3.2


### STAGE 2 - RAG PIPELINE

In [4]:
# LOAD PDF
documents = PyPDFLoader(PDF_FILE).load()
if not documents:
    raise ValueError(f"No pages were extracted from {PDF_FILE}. Check that the PDF is readable.")

# CHUNK DOCUMENTS
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = text_splitter.split_documents(documents)
if not chunks:
    raise ValueError("No text chunks were created from the PDF. Check whether the PDF contains selectable text.")

# CREATE FAISS VECTOR STORE
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# RAG PROMPT
rag_prompt = ChatPromptTemplate.from_template("""
Answer the question using only the retrieved context.
If the answer is not available in the context, say that the information could not be found in the documents.
Do not invent facts.

Context:
{context}

Question:
{question}

Answer:
""")

def get_context(query):
    retrieved_docs = retriever.invoke(query)
    retrieval_context = [doc.page_content for doc in retrieved_docs if doc.page_content.strip()]
    context = "\n\n".join(retrieval_context)
    return context, retrieval_context

print("PDF pages loaded:", len(documents))
print("Chunks created:", len(chunks))
print("FAISS vector store and top-3 retriever ready.")


PDF pages loaded: 6
Chunks created: 18
FAISS vector store and top-3 retriever ready.


### STAGE 3 - DEFINE EVALUATION METRICS

In [5]:
# SIMILARITY METRICS
bert_scorer = BERTScorer(lang="en", rescale_with_baseline=False)
rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

# ANSWER QUALITY METRICS
correctness_metric = GEval(
    name="Answer Correctness",
    model=evaluation_model,
    criteria=(
        "Compare the generated answer with the reference answer. Evaluate factual accuracy, "
        "important information, and contradictions. Ignore minor wording differences."
    ),
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT],
    async_mode=False,
)

helpfulness_metric = GEval(
    name="Helpfulness",
    model=evaluation_model,
    criteria=(
        "Evaluate whether the answer directly addresses the question, is easy for a beginner "
        "to understand, and gives a clear, useful explanation without unnecessary information."
    ),
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
    async_mode=False,
)

answer_relevancy_metric = AnswerRelevancyMetric(
    model=evaluation_model, include_reason=False, async_mode=False
)

# RAG METRICS
context_precision_metric = ContextualPrecisionMetric(model=evaluation_model, async_mode=False)
context_recall_metric = ContextualRecallMetric(model=evaluation_model, async_mode=False)
faithfulness_metric = FaithfulnessMetric(model=evaluation_model, async_mode=False)
rag_answer_relevancy_metric = AnswerRelevancyMetric(
    model=evaluation_model, include_reason=False, async_mode=False
)

rag_geval_metric = GEval(
    name="RAG Answer Correctness",
    model=evaluation_model,
    criteria=(
        "Evaluate whether the generated answer is factually correct and consistent with the "
        "reference answer. Ignore harmless wording differences."
    ),
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT],
    async_mode=False,
)

print("All DeepEval and text-similarity metrics initialized.")


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

All DeepEval and text-similarity metrics initialized.


### STAGE 4 - INPUT QUERIES, GENERATE ANSWERS AND CALCULATE OPERATIONAL METRICS

In [6]:
results = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="RAG generation"):
    query = row["User Query"]
    reference = row["Teacher Response"]
    total_start = time.perf_counter()
    retrieval_latency = generation_latency = np.nan

    try:
        start = time.perf_counter()
        context, retrieval_context = get_context(query)
        retrieval_latency = time.perf_counter() - start

        prompt_value = rag_prompt.invoke({"context": context, "question": query})
        start = time.perf_counter()
        response = llama_model.invoke(prompt_value)
        generation_latency = time.perf_counter() - start
        total_latency = time.perf_counter() - total_start
        generated = str(response.content).strip()

        metadata = getattr(response, "response_metadata", {}) or {}
        input_tokens = int(metadata.get("prompt_eval_count", 0) or 0)
        output_tokens = int(metadata.get("eval_count", 0) or 0)
        total_tokens = input_tokens + output_tokens

        input_cost = input_tokens * INPUT_PRICE
        output_cost = output_tokens * OUTPUT_PRICE

        results.append({
            "User Query": query,
            "Teacher Response": reference,
            "GeneratedAnswer": generated,
            "RetrievedContext": retrieval_context,
            "RetrievalLatency": retrieval_latency,
            "GenerationLatency": generation_latency,
            "Latency": total_latency,
            "InputTokens": input_tokens,
            "OutputTokens": output_tokens,
            "TotalTokens": total_tokens,
            "InputCost": input_cost,
            "OutputCost": output_cost,
            "TotalCost": input_cost + output_cost,
        })
    except Exception as e:
        print(f"\nGeneration error for query {query!r}: {e}")
        results.append({
            "User Query": query,
            "Teacher Response": reference,
            "GeneratedAnswer": "",
            "RetrievedContext": [],
            "RetrievalLatency": retrieval_latency,
            "GenerationLatency": generation_latency,
            "Latency": time.perf_counter() - total_start,
            "InputTokens": np.nan,
            "OutputTokens": np.nan,
            "TotalTokens": np.nan,
            "InputCost": np.nan,
            "OutputCost": np.nan,
            "TotalCost": np.nan,
        })

results_df = pd.DataFrame(results)
display(results_df)


RAG generation:   0%|          | 0/20 [00:00<?, ?it/s]

,User Query,Teacher Response,GeneratedAnswer,RetrievedContext,RetrievalLatency,GenerationLatency,Latency,InputTokens,OutputTokens,TotalTokens,InputCost,OutputCost,TotalCost
0,What is AI?,AI (Artificial Intelligence) is the ability of...,"According to the context, AI is the ability of...",[knowledge for a small educational chatbot or ...,0.060762,47.170181,47.231635,561,41,602,0.0,0.0,0.0
1,How does AI differ from traditional programming?,Traditional programs follow fixed rules writte...,AI differs from traditional programming mainly...,[knowledge for a small educational chatbot or ...,0.102364,13.444972,13.547879,560,56,616,0.0,0.0,0.0
2,What is Machine Learning?,Machine Learning is a branch of AI where compu...,Machine learning is a branch of artificial int...,[Traditional programming remains valuable when...,0.056403,15.207708,15.264400,471,59,530,0.0,0.0,0.0
3,What is Deep Learning?,Deep Learning is a type of Machine Learning th...,Deep learning is a specialized area within mac...,[Traditional programming remains valuable when...,0.206811,9.773090,9.980300,471,28,499,0.0,0.0,0.0
4,"What is the difference between AI, ML, and Dee...",AI is the broad field of making machines intel...,The information could not be found in the docu...,[Traditional programming remains valuable when...,0.108143,2.569394,2.677907,479,11,490,0.0,0.0,0.0
5,What is training data?,Training data is the data used to teach a Mach...,Training data is the collection of examples us...,[AI TEACHING DATASET - PARAGRAPH DOCUMENTATION...,0.199723,12.388343,12.588386,609,16,625,0.0,0.0,0.0
6,What is a Machine Learning model?,A Machine Learning model is a mathematical sys...,A Machine Learning model is the mathematical s...,[AI TEACHING DATASET - PARAGRAPH DOCUMENTATION...,0.096466,15.599886,15.696654,607,56,663,0.0,0.0,0.0
7,What is supervised learning?,Supervised learning trains a model using examp...,Supervised learning uses examples that contain...,"[recommendation, or decision. The model does n...",0.055284,14.511198,14.566800,608,32,640,0.0,0.0,0.0
8,What is unsupervised learning?,Unsupervised learning works with data that doe...,Unsupervised learning works with information t...,"[recommendation, or decision. The model does n...",0.041547,5.639110,5.680974,610,35,645,0.0,0.0,0.0
9,What is reinforcement learning?,Reinforcement learning teaches an agent throug...,Reinforcement learning teaches an agent throug...,"[model searches for hidden structures, similar...",0.113634,20.316689,20.430672,553,66,619,0.0,0.0,0.0


### STAGE 5 - LLM EVALUATION - ANSWER QUALITY

In [7]:
quality_scores = []

for _, row in tqdm(results_df.iterrows(), total=len(results_df), desc="Answer quality"):
    query = str(row["User Query"])
    generated = row["GeneratedAnswer"]
    reference = str(row["Teacher Response"])

    if not isinstance(generated, str) or not generated.strip():
        quality_scores.append([np.nan, np.nan, np.nan])
        continue

    test_case = LLMTestCase(
        input=query,
        actual_output=generated,
        expected_output=reference,
    )
    scores = []
    for metric in [correctness_metric, answer_relevancy_metric, helpfulness_metric]:
        try:
            metric.measure(test_case)
            scores.append(float(metric.score) if metric.score is not None else np.nan)
        except Exception as e:
            print(f"Query {query!r}, {metric.__class__.__name__} error: {e}")
            scores.append(np.nan)
    quality_scores.append(scores)

results_df[["AnswerCorrectness", "AnswerRelevancy", "Helpfulness"]] = pd.DataFrame(
    quality_scores, index=results_df.index, columns=["AnswerCorrectness", "AnswerRelevancy", "Helpfulness"]
)
results_df["AnswerQuality"] = results_df[["AnswerCorrectness", "AnswerRelevancy", "Helpfulness"]].mean(axis=1)

display(results_df[["User Query", "AnswerCorrectness", "AnswerRelevancy", "Helpfulness", "AnswerQuality"]])


Answer quality:   0%|          | 0/20 [00:00<?, ?it/s]

Output()

Output()

Output()

Output()

Output()

... (52 outputs left out)

Output()

Output()

Output()

,User Query,AnswerCorrectness,AnswerRelevancy,Helpfulness,AnswerQuality
0,What is AI?,0.8,1.000000,0.9,0.900000
1,How does AI differ from traditional programming?,0.4,0.666667,0.8,0.622222
2,What is Machine Learning?,0.6,1.000000,0.9,0.833333
3,What is Deep Learning?,0.6,1.000000,0.9,0.833333
4,"What is the difference between AI, ML, and Dee...",0.2,0.000000,0.0,0.066667
5,What is training data?,0.4,1.000000,0.6,0.666667
6,What is a Machine Learning model?,0.6,1.000000,0.3,0.633333
7,What is supervised learning?,0.6,1.000000,0.3,0.633333
8,What is unsupervised learning?,0.8,1.000000,0.4,0.733333
9,What is reinforcement learning?,0.6,1.000000,0.9,0.833333


### STAGE 6 - EVALUATE SIMILARITY METRICS

In [8]:
similarity_scores = []

for _, row in tqdm(
    results_df.iterrows(),
    total=len(results_df),
    desc = "Similarity metrics"):

    reference = row["Teacher Response"]
    generated = row["GeneratedAnswer"]

    if not generated:
        similarity_scores.append([np.nan] * 5)
        continue

    try:
        _, _, f1 = bert_scorer.score([generated], [reference])
        bert = float(f1[0])
    except Exception:
        bert = np.nan

    try:
        rouge_scores = rouge.score(reference, generated)
        rouge1 = rouge_scores["rouge1"].fmeasure
        rouge2 = rouge_scores["rouge2"].fmeasure
        rougeL = rouge_scores["rougeL"].fmeasure
    except Exception:
        rouge1 = rouge2 = rougeL = np.nan

    try:
        meteor = meteor_score([reference.split()], generated.split())
    except Exception:
        meteor = np.nan

    similarity_scores.append([bert, rouge1, rouge2, rougeL, meteor])

similarity_columns = ["BERT", "ROUGE-1", "ROUGE-2", "ROUGE-L", "METEOR"]

results_df[similarity_columns] = pd.DataFrame(similarity_scores, index = results_df.index)

display(results_df[["User Query"] + similarity_columns])

Similarity metrics:   0%|          | 0/20 [00:00<?, ?it/s]

,User Query,BERT,ROUGE-1,ROUGE-2,ROUGE-L,METEOR
0,What is AI?,0.954102,0.771930,0.545455,0.736842,0.842244
1,How does AI differ from traditional programming?,0.910140,0.438356,0.253521,0.410959,0.550044
2,What is Machine Learning?,0.909317,0.444444,0.278481,0.444444,0.457683
3,What is Deep Learning?,0.970048,0.800000,0.625000,0.800000,0.819990
4,"What is the difference between AI, ML, and Dee...",0.847488,0.047619,0.000000,0.047619,0.016835
5,What is training data?,0.926784,0.487805,0.410256,0.487805,0.364786
6,What is a Machine Learning model?,0.903768,0.434783,0.179104,0.347826,0.419154
7,What is supervised learning?,0.898333,0.384615,0.120000,0.269231,0.339549
8,What is unsupervised learning?,0.955249,0.627451,0.285714,0.509804,0.558511
9,What is reinforcement learning?,0.927300,0.459459,0.222222,0.351351,0.618849


### STAGE 7 - RAG EVALUATION METRICS

This stage evaluates retrieval quality (context precision/recall), groundedness (faithfulness), answer relevance, and answer correctness. Scores can be slow because each metric calls the local evaluation LLM.


In [9]:
import os

# Allow more time for each LLM call
os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "300"

# Avoid repeated attempts that can make evaluation slower
os.environ["DEEPEVAL_RETRY_MAX_ATTEMPTS"] = "1"

In [10]:
# Stage 7: evaluate retrieval and answer quality with the retrieved PDF context.
# Scores are checkpointed after every metric. A matching checkpoint is loaded when available.

rag_metric_columns = [
    "Context Precision",
    "Context Recall",
    "Faithfulness",
    "RAG Answer Relevancy",
    "RAG GEval",
]
rag_metrics = [
    context_precision_metric,
    context_recall_metric,
    faithfulness_metric,
    rag_answer_relevancy_metric,
    rag_geval_metric,
]
checkpoint_path = "rag_evaluation_checkpoint.csv"

for col in rag_metric_columns:
    if col not in results_df.columns:
        results_df[col] = np.nan

# Restore saved scores only when the checkpoint rows match the current input order.
if os.path.isfile(checkpoint_path):
    try:
        saved = pd.read_csv(checkpoint_path)
        same_rows = (
            len(saved) == len(results_df)
            and "User Query" in saved.columns
            and saved["User Query"].astype(str).tolist() == results_df["User Query"].astype(str).tolist()
            and "Teacher Response" in saved.columns
            and saved["Teacher Response"].astype(str).tolist() == results_df["Teacher Response"].astype(str).tolist()
        )
        if same_rows:
            for col in rag_metric_columns:
                if col in saved.columns:
                    results_df[col] = pd.to_numeric(saved[col], errors="coerce")
            print(f"Restored saved Stage 7 scores from {checkpoint_path}")
        else:
            print("Checkpoint ignored because its questions/references do not match this input dataset.")
    except Exception as e:
        print(f"Could not read checkpoint; starting Stage 7 without saved scores: {e}")

for idx, row in tqdm(results_df.iterrows(), total=len(results_df), desc="Stage 7: RAG evaluation"):
    generated = row.get("GeneratedAnswer", "")
    if not isinstance(generated, str) or not generated.strip():
        continue

    context = row.get("RetrievedContext", [])
    if isinstance(context, str):
        context = [context] if context.strip() else []
    elif not isinstance(context, list):
        context = []

    # Context-dependent metrics cannot be meaningful without retrieved context.
    test_case = LLMTestCase(
        input=str(row["User Query"]),
        actual_output=generated,
        expected_output=str(row["Teacher Response"]),
        retrieval_context=[str(item) for item in context if str(item).strip()],
    )

    for col, metric in zip(rag_metric_columns, rag_metrics):
        if pd.notna(results_df.at[idx, col]):
            continue
        try:
            metric.measure(test_case)
            score = metric.score
            results_df.at[idx, col] = float(score) if score is not None else np.nan
        except Exception as e:
            print(f"Row {idx} | {col} failed: {e}")
            results_df.at[idx, col] = np.nan
        finally:
            # Save progress even when one metric fails.
            results_df.to_csv(checkpoint_path, index=False, encoding="utf-8-sig")

results_df["RAG Evaluation Score"] = results_df[rag_metric_columns].mean(axis=1)

display(results_df[["User Query", *rag_metric_columns, "RAG Evaluation Score"]])


Restored saved Stage 7 scores from rag_evaluation_checkpoint.csv


Stage 7: RAG evaluation:   0%|          | 0/20 [00:00<?, ?it/s]

Output()

Output()

Row 0 | Context Precision failed: 1 validation error for Verdicts
  Invalid JSON: EOF while parsing a string at line 53 column 70 [type=json_invalid, input_value='{\n  "verdicts": [\n    ...or clear learning goals', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/json_invalid


Output()

... (54 outputs left out)

Output()

Row 19 | Context Precision failed: 1 validation error for Verdicts
  Invalid JSON: EOF while parsing an object at line 46 column 3 [type=json_invalid, input_value='{\n  "verdicts": [\n    ... AI in education."\n   ', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/json_invalid


Output()

,User Query,Context Precision,Context Recall,Faithfulness,RAG Answer Relevancy,RAG GEval,RAG Evaluation Score
0,What is AI?,NaN,1.0,NaN,1.000000,0.4,0.800000
1,How does AI differ from traditional programming?,1.0000,NaN,0.666667,0.666667,0.0,0.583333
2,What is Machine Learning?,1.0000,NaN,1.000000,1.000000,0.0,0.750000
3,What is Deep Learning?,1.0000,NaN,0.666667,1.000000,0.0,0.666667
4,"What is the difference between AI, ML, and Dee...",NaN,1.0,1.000000,0.250000,0.0,0.562500
5,What is training data?,1.0000,1.0,1.000000,1.000000,0.0,0.800000
6,What is a Machine Learning model?,1.0000,NaN,1.000000,1.000000,0.4,0.850000
7,What is supervised learning?,1.0000,NaN,1.000000,1.000000,0.0,0.750000
8,What is unsupervised learning?,1.0000,NaN,1.000000,1.000000,0.0,0.750000
9,What is reinforcement learning?,0.8875,NaN,0.700000,1.000000,0.4,0.746875


### STAGE 8 - OPERATIONAL METRICS SUMMARY

In [11]:
operational_columns = [
    "RetrievalLatency", "GenerationLatency", "Latency", "InputTokens", "OutputTokens",
    "TotalTokens", "InputCost", "OutputCost", "TotalCost",
]
operational_summary = pd.DataFrame({
    "Metric": operational_columns,
    "Average": [pd.to_numeric(results_df[c], errors="coerce").mean() for c in operational_columns],
    "Total": [pd.to_numeric(results_df[c], errors="coerce").sum(min_count=1) for c in operational_columns],
})
display(operational_summary.round(6))


,Metric,Average,Total
0,RetrievalLatency,0.083940,1.678806
1,GenerationLatency,12.959390,259.187794
2,Latency,13.043684,260.873677
3,InputTokens,529.500000,10590.000000
4,OutputTokens,35.600000,712.000000
5,TotalTokens,565.100000,11302.000000
6,InputCost,0.000000,0.000000
7,OutputCost,0.000000,0.000000
8,TotalCost,0.000000,0.000000


### STAGE 9 - FINAL EVALUATION TABLE SUMMARY

In [12]:
# Only include actual numeric evaluation scores in the score summary.
answer_quality_columns = ["AnswerCorrectness", "AnswerRelevancy", "Helpfulness", "AnswerQuality"]
similarity_columns = ["BERT", "ROUGE-1", "ROUGE-2", "ROUGE-L", "METEOR"]
rag_metric_columns = [
    "Context Precision", "Context Recall", "Faithfulness", "RAG Answer Relevancy", "RAG GEval"
]
score_columns = answer_quality_columns + similarity_columns + rag_metric_columns + ["RAG Evaluation Score"]

# Round numeric score columns only. Keep original answers and retrieved contexts unchanged.
for col in score_columns:
    if col in results_df.columns:
        results_df[col] = pd.to_numeric(results_df[col], errors="coerce").round(3)

final_columns = [
    "User Query", "Teacher Response", "GeneratedAnswer", "RetrievedContext",
    *score_columns, *operational_columns,
]
# Include only columns that exist and avoid accidental duplicates.
final_columns = list(dict.fromkeys(col for col in final_columns if col in results_df.columns))
final_df = results_df[final_columns].copy()
display(final_df)

summary_df = pd.DataFrame({
    "Metric": [col for col in score_columns if col in results_df.columns],
    "Average Score": [results_df[col].mean() for col in score_columns if col in results_df.columns],
})
summary_df["Average Score"] = summary_df["Average Score"].round(3)
display(summary_df)

# Save all deliverables
final_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
summary_df.to_csv("RAG_LLM_EVALUATION_SUMMARY.csv", index=False, encoding="utf-8-sig")
operational_summary.to_csv("RAG_LLM_OPERATIONAL_SUMMARY.csv", index=False, encoding="utf-8-sig")
print("Evaluation completed.")
print("Final results:", OUTPUT_CSV)
print("Evaluation summary: RAG_LLM_EVALUATION_SUMMARY.csv")
print("Operational summary: RAG_LLM_OPERATIONAL_SUMMARY.csv")


,User Query,Teacher Response,GeneratedAnswer,RetrievedContext,AnswerCorrectness,AnswerRelevancy,Helpfulness,AnswerQuality,BERT,ROUGE-1,...,RAG Evaluation Score,RetrievalLatency,GenerationLatency,Latency,InputTokens,OutputTokens,TotalTokens,InputCost,OutputCost,TotalCost
0,What is AI?,AI (Artificial Intelligence) is the ability of...,"According to the context, AI is the ability of...",[knowledge for a small educational chatbot or ...,0.8,1.000,0.9,0.900,0.954,0.772,...,0.800,0.060762,47.170181,47.231635,561,41,602,0.0,0.0,0.0
1,How does AI differ from traditional programming?,Traditional programs follow fixed rules writte...,AI differs from traditional programming mainly...,[knowledge for a small educational chatbot or ...,0.4,0.667,0.8,0.622,0.910,0.438,...,0.583,0.102364,13.444972,13.547879,560,56,616,0.0,0.0,0.0
2,What is Machine Learning?,Machine Learning is a branch of AI where compu...,Machine learning is a branch of artificial int...,[Traditional programming remains valuable when...,0.6,1.000,0.9,0.833,0.909,0.444,...,0.750,0.056403,15.207708,15.264400,471,59,530,0.0,0.0,0.0
3,What is Deep Learning?,Deep Learning is a type of Machine Learning th...,Deep learning is a specialized area within mac...,[Traditional programming remains valuable when...,0.6,1.000,0.9,0.833,0.970,0.800,...,0.667,0.206811,9.773090,9.980300,471,28,499,0.0,0.0,0.0
4,"What is the difference between AI, ML, and Dee...",AI is the broad field of making machines intel...,The information could not be found in the docu...,[Traditional programming remains valuable when...,0.2,0.000,0.0,0.067,0.847,0.048,...,0.562,0.108143,2.569394,2.677907,479,11,490,0.0,0.0,0.0
5,What is training data?,Training data is the data used to teach a Mach...,Training data is the collection of examples us...,[AI TEACHING DATASET - PARAGRAPH DOCUMENTATION...,0.4,1.000,0.6,0.667,0.927,0.488,...,0.800,0.199723,12.388343,12.588386,609,16,625,0.0,0.0,0.0
6,What is a Machine Learning model?,A Machine Learning model is a mathematical sys...,A Machine Learning model is the mathematical s...,[AI TEACHING DATASET - PARAGRAPH DOCUMENTATION...,0.6,1.000,0.3,0.633,0.904,0.435,...,0.850,0.096466,15.599886,15.696654,607,56,663,0.0,0.0,0.0
7,What is supervised learning?,Supervised learning trains a model using examp...,Supervised learning uses examples that contain...,"[recommendation, or decision. The model does n...",0.6,1.000,0.3,0.633,0.898,0.385,...,0.750,0.055284,14.511198,14.566800,608,32,640,0.0,0.0,0.0
8,What is unsupervised learning?,Unsupervised learning works with data that doe...,Unsupervised learning works with information t...,"[recommendation, or decision. The model does n...",0.8,1.000,0.4,0.733,0.955,0.627,...,0.750,0.041547,5.639110,5.680974,610,35,645,0.0,0.0,0.0
9,What is reinforcement learning?,Reinforcement learning teaches an agent throug...,Reinforcement learning teaches an agent throug...,"[model searches for hidden structures, similar...",0.6,1.000,0.9,0.833,0.927,0.459,...,0.747,0.113634,20.316689,20.430672,553,66,619,0.0,0.0,0.0


,Metric,Average Score
0,AnswerCorrectness,0.580
1,AnswerRelevancy,0.933
2,Helpfulness,0.705
3,AnswerQuality,0.739
4,BERT,0.925
5,ROUGE-1,0.507
6,ROUGE-2,0.324
7,ROUGE-L,0.473
8,METEOR,0.472
9,Context Precision,0.993


Evaluation completed.
Final results: RAG_LLM_FINAL_EVALUATION.csv
Evaluation summary: RAG_LLM_EVALUATION_SUMMARY.csv
Operational summary: RAG_LLM_OPERATIONAL_SUMMARY.csv


In [2]:
# numpy: 1.26.4
# pandas: 2.2.2
# nltk: 3.10.0
# instructor: 1.17.0
# langchain-ollama: 1.1.0
# langchain-community: 0.4.1
# langchain-text-splitters: 1.1.3
# langchain-core: 1.7.0
# faiss-cpu: 1.15.1
# bert-score: 0.3.13
# rouge-score: 0.1.2
# ragas: 0.4.3
# deepeval: 4.3.1
# tqdm: 4.69.0